# EDA Inicial — Predicción de Retrasos en Entregas Logísticas

**Proyecto Final - MLOps** · Universidad de Medellín · Equipo 7

---

**Objetivo:**  predecir `Late_delivery_risk`, revisando estructura, integridad, distribuciones clave y riesgo de fuga de información.


## 1. Importación de Librerías

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import chi2_contingency

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)


## 2. Carga y Vista General de los Datos

In [5]:
DATA_PATH = Path('data/raw/supply_chain_data.csv')

df = pd.read_csv(DATA_PATH, encoding='latin-1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(10)


Dataset cargado: 100 filas × 24 columnas


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632
5,haircare,SKU5,1.699976,87,147,2828.348746,Non-binary,90,27,66,...,Bangalore,10,104,17,56.766476,Fail,2.779194,Road,Route A,235.461237
6,skincare,SKU6,4.078333,48,65,7823.476560,Male,11,15,58,...,Kolkata,14,314,24,1.085069,Pending,1.000911,Sea,Route A,134.369097
7,cosmetics,SKU7,42.958384,59,426,8496.103813,Female,93,17,11,...,Bangalore,22,564,1,99.466109,Fail,0.398177,Road,Route C,802.056312
8,cosmetics,SKU8,68.717597,78,150,7517.363211,Female,5,10,15,...,Mumbai,13,769,8,11.423027,Pending,2.709863,Sea,Route B,505.557134
9,skincare,SKU9,64.015733,35,980,4971.145988,Unknown,14,27,83,...,Chennai,29,963,23,47.957602,Pending,3.844614,Rail,Route B,995.929461


In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    object 
 1   SKU                      100 non-null    object 
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    object 
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    object 
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    object 
 14  Location                 10

In [7]:
df.describe(include='all')


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
count,100,100,100.000000,100.000000,100.000000,100.000000,100,100.000000,100.000000,100.000000,...,100,100.000000,100.000000,100.00000,100.000000,100,100.000000,100,100,100.000000
unique,3,100,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,...,5,NaN,NaN,NaN,NaN,3,NaN,4,3,NaN
top,skincare,SKU0,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,Kolkata,NaN,NaN,NaN,NaN,Pending,NaN,Road,Route A,NaN
freq,40,1,NaN,NaN,NaN,NaN,31,NaN,NaN,NaN,...,25,NaN,NaN,NaN,NaN,41,NaN,29,43,NaN
mean,NaN,NaN,49.462461,48.400000,460.990000,5776.048187,NaN,47.770000,15.960000,49.220000,...,NaN,17.080000,567.840000,14.77000,47.266693,NaN,2.277158,NaN,NaN,529.245782
std,NaN,NaN,31.168193,30.743317,303.780074,2732.841744,NaN,31.369372,8.785801,26.784429,...,NaN,8.846251,263.046861,8.91243,28.982841,NaN,1.461366,NaN,NaN,258.301696
min,NaN,NaN,1.699976,1.000000,8.000000,1061.618523,NaN,0.000000,1.000000,1.000000,...,NaN,1.000000,104.000000,1.00000,1.085069,NaN,0.018608,NaN,NaN,103.916248
25%,NaN,NaN,19.597823,22.750000,184.250000,2812.847151,NaN,16.750000,8.000000,26.000000,...,NaN,10.000000,352.000000,7.00000,22.983299,NaN,1.009650,NaN,NaN,318.778455
50%,NaN,NaN,51.239831,43.500000,392.500000,6006.352023,NaN,47.500000,17.000000,52.000000,...,NaN,18.000000,568.500000,14.00000,45.905622,NaN,2.141863,NaN,NaN,520.430444
75%,NaN,NaN,77.198228,75.000000,704.250000,8253.976921,NaN,73.000000,24.000000,71.250000,...,NaN,25.000000,797.000000,23.00000,68.621026,NaN,3.563995,NaN,NaN,763.078231


## 3. Limpieza de Datos

In [9]:
# Nulos y duplicados
print('=== Nulos ===')
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
with_nulls = pd.DataFrame({'nulos': nulls, 'pct_%': null_pct})
print(with_nulls[with_nulls['nulos'] > 0].sort_values('pct_%', ascending=False).to_string())

print(f'\n=== Duplicados ===')
print(f'Filas duplicadas: {df.duplicated().sum()}')

if 'Order Id' in df.columns:
    print(f'Órdenes únicas (Order Id): {df["Order Id"].nunique()} de {len(df)} filas (dataset a nivel ítem)')
elif 'SKU' in df.columns:
    print(f'SKU únicos: {df["SKU"].nunique()} de {len(df)} filas')
else:
    print('No se encontró una columna de identificador de orden (Order Id) en este dataset.')


=== Nulos ===
Empty DataFrame
Columns: [nulos, pct_%]
Index: []

=== Duplicados ===
Filas duplicadas: 0
SKU únicos: 100 de 100 filas


**Observaciones:** Las variables operativasno tienen nulos. Solo `Product Description` (100%) y `Order Zipcode` (~86%) son problemáticas — ambas se descartan. No hay filas duplicadas.

## 4. Análisis de la Variable Objetivo (`Late_delivery_risk`)

In [10]:
# Distribución de variable objetivo (si existe)
target_col = 'Late_delivery_risk'
if target_col in df.columns:
    target_counts = df[target_col].value_counts().sort_index()
    target_pct = df[target_col].value_counts(normalize=True).sort_index() * 100

    labels = {0: 'A tiempo (0)', 1: 'Riesgo de retraso (1)'}
    x_labels = [labels.get(k, str(k)) for k in target_counts.index]

    fig = px.bar(
        x=x_labels,
        y=target_counts.values,
        color=x_labels,
        color_discrete_sequence=px.colors.qualitative.Set2,
        title='Distribución de la Variable Objetivo: Late_delivery_risk',
        labels={'x': 'Estado de entrega', 'y': 'Número de registros'},
        text=target_counts.values
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(showlegend=False)
    fig.show()

    print('\nDistribución del target:')
    for k in target_counts.index:
        print(f'  {labels.get(k, str(k))}: {target_counts[k]:,} registros ({target_pct[k]:.1f}%)')
else:
    print("La columna 'Late_delivery_risk' no existe en este dataset.")
    print('Columnas disponibles:')
    print(', '.join(df.columns))


La columna 'Late_delivery_risk' no existe en este dataset.
Columnas disponibles:
Product type, SKU, Price, Availability, Number of products sold, Revenue generated, Customer demographics, Stock levels, Lead times, Order quantities, Shipping times, Shipping carriers, Shipping costs, Supplier name, Location, Lead time, Production volumes, Manufacturing lead time, Manufacturing costs, Inspection results, Defect rates, Transportation modes, Routes, Costs


##  Matriz de Correlación

In [11]:
df_corr = df.copy()

# Matriz de correlación para columnas numéricas disponibles
numeric_cols = df_corr.select_dtypes(include=np.number).columns.tolist()

if len(numeric_cols) >= 2:
    corr_matrix = df_corr[numeric_cols].corr()
    fig = px.imshow(
        corr_matrix,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Matriz de Correlación (Variables Numéricas)',
        text_auto='.2f',
        aspect='auto'
    )
    fig.update_layout(height=700)
    fig.show()
else:
    corr_matrix = pd.DataFrame()
    print('No hay suficientes columnas numéricas para calcular correlación.')


In [12]:
# Correlaciones con el target (si existe)
target_col = 'Late_delivery_risk'

if not corr_matrix.empty and target_col in corr_matrix.columns:
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
    print('Correlación con Late_delivery_risk:')
    print(target_corr.round(3).to_string())
else:
    print("No se puede calcular correlación con 'Late_delivery_risk' porque no está disponible en las columnas numéricas.")


No se puede calcular correlación con 'Late_delivery_risk' porque no está disponible en las columnas numéricas.


In [13]:
# Distribución geográfica / por ubicación
if 'Order Country' in df.columns:
    country_counts = df['Order Country'].value_counts().reset_index()
    country_counts.columns = ['País', 'Pedidos']

    fig_map = px.choropleth(
        country_counts,
        locations='País',
        locationmode='country names',
        color='Pedidos',
        color_continuous_scale='YlOrRd',
        title='Distribución Global de Pedidos por País',
        labels={'Pedidos': 'Número de Pedidos'}
    )
    fig_map.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        height=500
    )
    fig_map.show()

    print(f'\nTotal de países con pedidos: {country_counts["País"].nunique()}')
    print('\nTop 10 países por volumen:')
    print(country_counts.head(10).to_string(index=False))
elif 'Location' in df.columns:
    loc_counts = df['Location'].value_counts().reset_index()
    loc_counts.columns = ['Ubicación', 'Registros']

    fig = px.bar(
        loc_counts.head(15),
        x='Ubicación',
        y='Registros',
        color='Registros',
        color_continuous_scale='YlOrRd',
        title='Top 15 Ubicaciones por Número de Registros'
    )
    fig.update_layout(xaxis_title='Ubicación', yaxis_title='Número de registros')
    fig.show()

    print(f'\nTotal de ubicaciones: {loc_counts["Ubicación"].nunique()}')
    print('\nTop 10 ubicaciones por volumen:')
    print(loc_counts.head(10).to_string(index=False))
else:
    print("No se encontró columna geográfica ('Order Country' o 'Location').")



Total de ubicaciones: 5

Top 10 ubicaciones por volumen:
Ubicación  Registros
  Kolkata         25
   Mumbai         22
  Chennai         20
Bangalore         18
    Delhi         15
